# Underwater Object Detection — Demo
YOLOv5 model trained on the [aquarium-data-cots](https://www.kaggle.com/datasets/slavkoprytula/aquarium-data-cots) dataset (7 classes: fish, jellyfish, penguin, puffin, shark, starfish, stingray).

This notebook loads the best model from the **uwd-p3-final** training kernel (YOLOv5 champion config trained on train+val), runs detection on sample test images, and reports the official test-split metrics.

In [ ]:
import glob, json, os, shutil, sys

def load_src_library():
    dst = '/tmp/lib/src'
    os.makedirs(dst, exist_ok=True)
    roots = [p for p in glob.glob('/kaggle/input/**/uwd-src*', recursive=True) if os.path.isdir(p)]
    roots.sort(key=len)
    src_root = roots[0] if roots else '/kaggle/input'
    for root, _, files in os.walk(src_root):
        for f in files:
            if f.endswith('.py'):
                shutil.copy(os.path.join(root, f), os.path.join(dst, f))
            elif f.endswith('.zip'):
                shutil.unpack_archive(os.path.join(root, f), '/tmp/lib/unzipped')
    for root, _, files in os.walk('/tmp/lib/unzipped'):
        for f in files:
            if f.endswith('.py'):
                shutil.copy(os.path.join(root, f), dst)
    sys.path.insert(0, dst)

load_src_library()
from uwd_utils import find_under_input

BEST = find_under_input('best.pt')
assert BEST, 'best.pt not found — attach the uwd-p3-final kernel output'
STATS = find_under_input('p0_stats.json')
P0 = json.load(open(STATS)) if STATS else {}
print('model:', BEST)
print('classes:', P0.get('class_names'))

In [ ]:
# Load YOLOv5 (local clone — torch.hub's GitHub flow prompts interactively)
import subprocess, sys, os, pathlib
if not os.path.isdir('/tmp/yolov5'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ultralytics/yolov5', '/tmp/yolov5'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/tmp/yolov5/requirements.txt'])
# pre-place font so yolov5 never does a single-shot font download mid-run
font_dir = pathlib.Path.home() / '.config' / 'Ultralytics'
font_dir.mkdir(parents=True, exist_ok=True)
(font_dir / 'Arial.ttf').write_bytes(b'')  # PIL falls back to its default font
import torch
model = torch.hub.load('/tmp/yolov5', 'custom', source='local', path=BEST, device=0)
model.conf = 0.25
print(model)

In [ ]:
# Detection grid on sample test images
import matplotlib.pyplot as plt
from PIL import Image

test_dir = os.path.join(P0['split_dirs']['test'], 'images')
samples = sorted(glob.glob(os.path.join(test_dir, '*')))[:12]
fig, axes = plt.subplots(3, 4, figsize=(22, 14))
for ax, ip in zip(axes.ravel(), samples):
    im = Image.open(ip)
    res = model(im, size=640)
    res.render()
    ax.imshow(res.ims[0])
    n = len(res.pred[0])
    ax.set_title(f'{os.path.basename(ip)} — {n} det.', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/detections_grid.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# Official test-split metrics (from the final training kernel)
m = find_under_input('p3_final_metrics.json')
if m:
    met = json.load(open(m))
    for key in ('test', 'test_tta'):
        rows = met.get(key) or []
        if rows:
            allr = rows[0]
            print(f"{key}: P={allr['P']:.4f} R={allr['R']:.4f} "
                  f"mAP@0.5={allr['mAP50']:.4f} mAP@0.5:0.95={allr['mAP5095']:.4f}")
            print('  per-class mAP@0.5:', {r['cls']: r['mAP50'] for r in rows[1:]})
else:
    print('p3_final_metrics.json not attached; run val below instead')

In [ ]:
# Re-verify on the test split (no TTA) directly in this notebook
DATA = find_under_input('data_fixed.yaml')
if DATA:
    import subprocess, sys as _sys
    import glob as _g
    yv5 = '/tmp/yolov5'
    if not os.path.isdir(yv5):
        import subprocess as sp
        sp.run(['git', 'clone', '-q', 'https://github.com/ultralytics/yolov5', yv5], check=True)
        sp.run([_sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{yv5}/requirements.txt'])
    r = subprocess.run([_sys.executable, f'{yv5}/val.py', '--weights', BEST,
                        '--data', DATA, '--task', 'test', '--img', '640',
                        '--project', '/kaggle/working/val', '--name', 'demo',
                        '--device', '0', '--exist-ok'], capture_output=True, text=True)
    print(r.stdout[-2000:])
else:
    print('data_fixed.yaml not attached (needs uwd-p0-smoke output)')